# Dynamic Configuration

Most of the time, we are not actually interested in just running a 
simulation that we've pre-configured in files on disk. Instead, we will want to load 
a simulation from pre-configured in files on disk, but then change a couple things
before we run it.  Perhaps we're debugging a new RM system, and want to just run
a few iterations to see if it works.  Or maybe we have a model mostly ready to go,
but we want to modify a couple of RM system parameters, or change the level of demand,
or just monkey around with the network schedule a bit.

We can do all of that and more, right here in Python, especially in a Jupyter notebook.

In [ ]:
import passengersim as pax

pax.versions()

Let's start by loading our demo model from its YAML files into a
PassengerSim `Config` object.

In [ ]:
cfg = pax.Config.from_yaml(pax.demo_network("3MKT/08-untrunc-em"))

The `cfg` now holds the data that defines everything about the simulation that will be run,
including the carriers, RM systems, networks, ... everything.  Before we use this config to
initialize a simulation, we can inspect and even modify it in Python.  For example, let's take
a look at the flight legs in this network.

In [ ]:
cfg.legs

We can see here a list of legs, and each leg has all the attributes needed to define it
in the simulation: origin, destination, departure and arrival times, capacity, and more.
We can manipulate these attributes if we like.  Let's make the capacity on that first leg
a little larger.

In [ ]:
cfg.legs[0].capacity = 180

We can do the same and peek in on the network fares.

In [ ]:
cfg.fares

There are a lot of fares for our tiny little network.  Perhaps we don't want to edit them one at a time.
We can do a little Python programming to modify fares all at once according to some logic.  We put a bigger
plane on one of AL1's Boston to Chicago legs, so let's lower their fares in that market a smidge to help fill
that plane.

In [ ]:
for f in cfg.fares:
    if f.carrier == "AL1" and f.orig == "BOS" and f.dest == "ORD":
        f.price -= 5.0

Not all our changes need to be about the network; we can also manipulate the settings
for the simulation itself.  Our modified network is just a small experiment, we we 
don't need to sit around waiting for a statistically useful number of samples to run.
Let's cut way back so our simulation runs faster.

In [ ]:
cfg.simulation_controls.num_trials = 2
cfg.simulation_controls.num_samples = 200

Now that we've modified our `cfg` all we want, we can use it to 
initialize a simulation to run.

In [ ]:
sim = pax.Simulation(cfg)
summary = sim.run()

Having run the simulation, we can take a quick peek at the results.

In [ ]:
summary.fig_carrier_revenues()

In [ ]:
summary.fig_carrier_rasm()

In [ ]:
summary.fig_fare_class_mix()